# MTG-I (Meteosat Third Generation Imager) / FCI (Flexible Combined Imager)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=mtg/mtg-fci). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1c

Level 1c: calibrated, geolocated and harmonised radiances/reflectances for the 16 channels, as full-disc high spectral resolution (FDHSI, normal resolution) or high-resolution fast imagery (HRFI) NetCDF files in chunks.

Downloads one FCI full-disc cycle (normal resolution, EO:EUM:DAT:0662) from the EUMETSAT Data Store (needs a free account and consumer key/secret, asked at run time) and plots the 10.5 um channel with satpy.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:0662")

products = list(collection.search(dtstart=dt.datetime(2025, 6, 1, 12, 0), dtend=dt.datetime(2025, 6, 1, 12, 10)))
print(len(products), "FCI L1c products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# about 1-2 GB per full-disc cycle: the download can take several minutes
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

files = sorted(glob("data/**/*.nc", recursive=True))
print(len(files), "files")
scn = Scene(filenames=files, reader="fci_l1c_nc")
print(scn.available_dataset_names())
scn.load(["ir_105"])
scn["ir_105"].plot(cmap="gray_r")               # brightness temperature (K)
plt.title("FCI IR 10.5 um")
plt.show()

## L2

Level 2: products from FCI: cloud mask (CLM), all-sky radiance (ASR), global instability index (GII), optimal cloud analysis (OCA), atmospheric motion vectors (AMV) and others.

Downloads one FCI Level-2 cloud mask (netCDF) from the EUMETSAT Data Store (needs a free account and consumer key/secret, asked at run time) and plots the first dataset with satpy.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:0678")

products = list(collection.search(dtstart=dt.datetime(2025, 6, 1, 12, 0), dtend=dt.datetime(2025, 6, 1, 12, 20)))
print(len(products), "FCI cloud mask products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# tens of MB per product
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

files = sorted(glob("data/**/*.nc", recursive=True))
print(len(files), "files")
scn = Scene(filenames=files, reader="fci_l2_nc")
print(scn.available_dataset_names())
name = scn.available_dataset_names()[0]
scn.load([name])
scn[name].plot()
plt.title(name)
plt.show()